# Hyperparameter Search for $\lambda_{obs}$ vs $\alpha_{train}$

This notebook searches $\lambda_{obs}$ for each $\alpha_{train} \in \{1.0, 0.2, 0.05\}$.

For each alpha: start from the same small $\lambda_{obs}$, train, evaluate the 95th quantile of maximum violation on the validation set, and update $\lambda_{obs}$:
- if quantile > 0: increase $\lambda_{obs}$
- else: decrease $\lambda_{obs}$

Goal: find a $\lambda_{obs}$ with violation quantile close to 0.

In [6]:
import torch
import numpy as np
from dataclasses import dataclass, field

from performance_boosting import PBClosedLoop
from ren import ContractiveREN
from robot import RobotPlant, StabilizedRobot, PDController
from dataset import generate_random_batch
from losses_and_wrappers import PBLoss, SplitCVaRLossWrapper
from training_function import train_agent
from wakepy import keep

In [7]:
from __future__ import annotations

def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)


@dataclass
class ExperimentConfig:
    # --- 1. General Setup ---
    device: torch.device = field(default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    seed: int = 4

    # --- 2. Dataset Parameters ---
    noise_std: float = 0.0

    # --- 3. Physics & Dynamics Parameters ---
    n_agents: int = 1
    state_dim: int = 4
    input_dim: int = 2
    dt: float = 0.05

    b_nom: float = 1.0
    m_nom: float = 1.0
    b2_nom: float = 0.2

    b_sim: float = 1.0
    m_sim: float = 1.0
    b2_sim: float = 0.2

    # --- 4. REN and Base Controller Parameters ---
    initialization_std: float = 0.01
    dim_internal: int = 4
    dim_nl: int = 4
    kp: float = 1.0
    ki: float = 1.0

    # --- 5. Loss & Cost Parameters ---
    lambda_x: float = 10.0
    lambda_u: float = 0.0
    lambda_decoupling: float = 1.0
    lambda_obs: float = 1.0
    tau_safe_bar: float = 0.0
    track_mode: str = 'quadratic'
    coll_mode: str = 'rbf'

    # --- 6. Wrapper Parameters ---
    alpha_train: float = 0.2

    # --- 7. Training & Validation Parameters ---
    num_training_steps: int = 2000
    n_inner_steps: int = 5
    log_interval: int = 1
    early_stopping_patience_limit: int | None = None
    gradient_clipping: float | None = None
    batch_size: int = 500
    horizon: int = 500
    lr: float = 0.001
    num_val_samples: int = 500

    def __post_init__(self):
        self.x0_centers = [[-1.0, -1.0], [1.0, -1.0], [1.0, 1.5]]
        self.x0_stds = [0.2, 0.1, 0.4]
        self.x0_probs = [0.6, 0.2, 0.2]

        self.Q_agent = torch.diag(torch.tensor([1.0, 1.0, 1.0, 1.0])) * self.lambda_x
        self.Q = torch.kron(torch.eye(self.n_agents), self.Q_agent).to(self.device)

        self.R_agent = torch.eye(2) * self.lambda_u
        self.R = torch.kron(torch.eye(self.n_agents), self.R_agent).to(self.device)

        self.obs_centers = [torch.tensor([0.5, -0.5]).to(self.device)]
        self.obs_radii = [[0.8, 0.15]]
        self.safety_margin = 0.05
        self.obs_radii_safe = [[r + self.safety_margin for r in obs] for obs in self.obs_radii]

        self.x_target = torch.zeros(4 * self.n_agents).to(self.device)


def build_training_objects(config: ExperimentConfig):
    sim_OL_plant = RobotPlant(b=config.b_sim, b2=config.b2_sim, m=config.m_sim, n_agents=config.n_agents).to(config.device)
    nominal_OL_plant = RobotPlant(b=config.b_nom, b2=config.b2_nom, m=config.m_nom, n_agents=config.n_agents).to(config.device)
    base_controller = PDController(kp=config.kp, ki=config.ki, n_agents=config.n_agents).to(config.device)

    f_sim = StabilizedRobot(sim_OL_plant, base_controller).to(config.device)
    f_nom = StabilizedRobot(nominal_OL_plant, base_controller).to(config.device)

    for param in f_sim.parameters():
        param.requires_grad = False
    for param in f_nom.parameters():
        param.requires_grad = False

    ren = ContractiveREN(
        dim_in=config.state_dim * config.n_agents,
        dim_out=config.input_dim * config.n_agents,
        dim_internal=config.dim_internal,
        dim_nl=config.dim_nl,
        initialization_std=config.initialization_std,
    ).to(config.device)

    sim = PBClosedLoop(ren, f_sim, f_nom).to(config.device)

    metric = PBLoss(
        x_target=config.x_target,
        Q=config.Q,
        R=config.R,
        lambda_obs=config.lambda_obs,
        obs_centers=config.obs_centers,
        obs_radii_safe=config.obs_radii_safe,
        n_agents=config.n_agents,
        track_mode=config.track_mode,
        coll_mode=config.coll_mode,
    ).to(config.device)

    wrapper = SplitCVaRLossWrapper(alpha=config.alpha_train, metric=metric, lambda_decoupling=config.lambda_decoupling).to(config.device)

    return sim, metric, wrapper


def compute_maximum_violation(traj_x: torch.Tensor, config: ExperimentConfig) -> torch.Tensor:
    x_reshaped = traj_x.view(traj_x.shape[0], traj_x.shape[1], config.n_agents, 4)
    pos = x_reshaped[..., :2]  # [B, H, N, 2]

    obs_centers_t = torch.stack(config.obs_centers).to(config.device)
    obs_radii_safe_t = torch.tensor(config.obs_radii_safe, dtype=torch.float32, device=config.device)

    centers = obs_centers_t.view(1, 1, 1, -1, 2)
    diff = pos.unsqueeze(3) - centers
    r_safe = obs_radii_safe_t.view(1, 1, 1, -1, 2)

    eps = 1e-12
    rho = torch.linalg.norm(diff / (r_safe + eps), dim=-1)
    rho_safe = torch.clamp(rho, min=eps)
    border_point = centers + diff / rho_safe.unsqueeze(-1)

    unsigned_dist = torch.linalg.norm(pos.unsqueeze(3) - border_point, dim=-1)
    signed_dist = torch.where(rho < 1.0, unsigned_dist, -unsigned_dist)

    min_safe_radius = r_safe.min(dim=-1).values.expand_as(signed_dist)
    signed_dist = torch.where(rho < 1e-9, min_safe_radius, signed_dist)

    maximum_violation = signed_dist.amax(dim=(1, 2, 3))
    return maximum_violation


def evaluate_model(sim, metric, fixed_val_w, config: ExperimentConfig):
    sim.eval()
    with torch.no_grad():
        traj_x_val, traj_u_val, _ = sim.run(fixed_val_w)
        _, cost_x_val, cost_u_val, _ = metric(traj_x_val, traj_u_val)

    perf_qr = cost_x_val + cost_u_val
    maximum_violation = compute_maximum_violation(traj_x_val, config)

    return {
        'mean_perf_qr': perf_qr.mean().item(),
        'violation_quantile_95': torch.quantile(maximum_violation, 0.95).item(),
        'violation_rate': (maximum_violation > 0.0).float().mean().item(),
    }

In [8]:
# Search settings
alpha_train_values = [1.0, 0.2, 0.05]
initial_lambda_obs = 100.0
lambda_mult_up = 2.0
lambda_mult_down = 0.5
max_search_iters = 20

# Build one fixed validation batch shared across all runs
set_seed(4)
base_cfg = ExperimentConfig(alpha_train=alpha_train_values[0], lambda_obs=initial_lambda_obs)
fixed_val_w = generate_random_batch(base_cfg, custom_batch_size=base_cfg.num_val_samples)

search_results = {}

In [ ]:
for alpha in alpha_train_values:
    print("=" * 90)
    print(f"Searching lambda_obs for alpha_train={alpha}")

    current_lambda = initial_lambda_obs
    best = None
    last_negative = None
    previous_direction = None
    stop_reason = "max_search_iters reached"

    for it in range(1, max_search_iters + 1):
        set_seed(4 + it)
        cfg = ExperimentConfig(alpha_train=alpha, lambda_obs=current_lambda)

        sim, metric, wrapper = build_training_objects(cfg)

        with keep.presenting():
            history, _ = train_agent(
                config=cfg,
                sim=sim,
                loss_wrapper=wrapper,
                mode="standard_cvar",
                fixed_val_w=fixed_val_w,
                generate_random_batch=generate_random_batch,
                plot_results=False,
                plot_kwargs=None,
            )

        metrics = evaluate_model(sim, metric, fixed_val_w, cfg)
        q95 = metrics['violation_quantile_95']
        abs_q95 = abs(q95)

        candidate = {
            'alpha_train': alpha,
            'lambda_obs': current_lambda,
            'mean_perf_qr': metrics['mean_perf_qr'],
            'violation_quantile_95': q95,
            'violation_rate': metrics['violation_rate'],
            'abs_q95': abs_q95,
            'search_iter': it,
        }

        if (best is None) or (abs_q95 < best['abs_q95']):
            best = candidate

        if q95 < 0.0:
            last_negative = candidate

        print(
            f"iter={it:02d} | lambda_obs={current_lambda:.6f} | "
            f"q95={q95:.6f} | perf={metrics['mean_perf_qr']:.6f} | "
            f"viol_rate={metrics['violation_rate']:.2%}"
        )

        direction = "increase" if q95 > 0.0 else "decrease"

        if previous_direction is not None and direction != previous_direction:
            if last_negative is not None:
                best = last_negative
                stop_reason = (
                    f"direction flip detected ({previous_direction} -> {direction}); "
                    "retained last negative q95"
                )
            else:
                stop_reason = (
                    f"direction flip detected ({previous_direction} -> {direction}); "
                    "no negative q95 seen, kept best abs(q95)"
                )
            break

        previous_direction = direction

        if direction == "increase":
            current_lambda = min(current_lambda * lambda_mult_up, 1e6)
        else:
            current_lambda = max(current_lambda * lambda_mult_down, 1e-8)

    search_results[alpha] = best
    print(
        f"Selected lambda_obs={best['lambda_obs']:.6f} for alpha_train={alpha} "
        f"(best q95={best['violation_quantile_95']:.6f}; stop: {stop_reason})"
    )

print("\nSearch completed for all alpha values.")

Searching lambda_obs for alpha_train=1.0
Starting STANDARD_CVAR online training on cpu...


Standard Cvar: 100%|██████████| 2000/2000 [17:52<00:00,  1.86it/s, Val Loss=1.2080, Best=1.2061]



Restored best model (Metric: 1.2061).
iter=01 | lambda_obs=100.000000 | q95=0.123795 | perf=1.117496 | viol_rate=15.60%
Starting STANDARD_CVAR online training on cpu...


Standard Cvar: 100%|██████████| 2000/2000 [17:58<00:00,  1.85it/s, Val Loss=1.2731, Best=1.2733]



Restored best model (Metric: 1.2731).
iter=02 | lambda_obs=200.000000 | q95=0.365982 | perf=1.126583 | viol_rate=17.80%
Starting STANDARD_CVAR online training on cpu...


...


Restored best model (Metric: 6.3049).
iter=06 | lambda_obs=3200.000000 | q95=0.178952 | perf=1.736584 | viol_rate=16.40%
Starting STANDARD_CVAR online training on cpu...


Standard Cvar: 100%|██████████| 2000/2000 [18:10<00:00,  1.83it/s, Val Loss=12.4237, Best=10.6247] 



Restored best model (Metric: 10.6247).
iter=07 | lambda_obs=6400.000000 | q95=0.152396 | perf=1.792053 | viol_rate=15.80%
Starting STANDARD_CVAR online training on cpu...


Standard Cvar: 100%|██████████| 2000/2000 [18:10<00:00,  1.83it/s, Val Loss=6.9590, Best=6.0403]    



Restored best model (Metric: 6.0403).
iter=08 | lambda_obs=12800.000000 | q95=-0.260471 | perf=3.685382 | viol_rate=0.00%
Selected lambda_obs=6400.000000 for alpha_train=0.05 (best q95=0.152396; stop: direction flip detected (increase -> decrease))

Search completed for all alpha values.


In [10]:
print("Final comparison (using best lambda_obs per alpha_train):")
print("-" * 90)
print(f"{'alpha_train':>12} | {'lambda_obs':>12} | {'mean_perf_qr':>14} | {'viol_q95':>12} | {'viol_rate':>10}")
print("-" * 90)

for alpha in alpha_train_values:
    r = search_results[alpha]
    print(
        f"{r['alpha_train']:>12.3f} | {r['lambda_obs']:>12.6f} | {r['mean_perf_qr']:>14.6f} | "
        f"{r['violation_quantile_95']:>12.6f} | {r['violation_rate']:>9.2%}"
    )

print("-" * 90)

Final comparison (using best lambda_obs per alpha_train):
------------------------------------------------------------------------------------------
 alpha_train |   lambda_obs |   mean_perf_qr |     viol_q95 |  viol_rate
------------------------------------------------------------------------------------------
       1.000 |  3200.000000 |       1.604676 |    -0.029664 |     3.80%
       0.200 |  3200.000000 |       1.909883 |    -0.046482 |     3.20%
       0.050 |  6400.000000 |       1.792053 |     0.152396 |    15.80%
------------------------------------------------------------------------------------------
